<a href="https://colab.research.google.com/github/yunsoyoung2004/TTMchatbot/blob/main/%EA%B2%BD%EB%9F%89%ED%99%94_%ED%8B%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 완전 정리 및 최신 설치
!pip uninstall -y transformers accelerate trl
!pip install -U transformers accelerate datasets peft bitsandbytes trl

# 2. 버전 확인
import transformers, accelerate, trl
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)

Found existing installation: transformers 4.52.0.dev0
Uninstalling transformers-4.52.0.dev0:
  Successfully uninstalled transformers-4.52.0.dev0
Found existing installation: accelerate 1.6.0
Uninstalling accelerate-1.6.0:
  Successfully uninstalled accelerate-1.6.0
Found existing installation: trl 0.17.0
Uninstalling trl-0.17.0:
  Successfully uninstalled trl-0.17.0
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached trl-0.17.0-py3-none-any.whl.metadata (12 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.6.0-py3-none-any.whl (354 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 7.6 MB/s eta 0:00:00
Using cached trl-0.17.0-py3-none-any.whl (348 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.15.2.dev0
    Uninstalling peft-0.15.2.dev0:
      Successfully uninstalled peft-0.15.2.dev0
transformers: 4.51.3
acceler

In [ ]:
import transformers
print("Transformers version:", transformers.__version__)


Transformers version: 4.51.3


In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model

# ✅ CUDA 캐시 초기화
torch.cuda.empty_cache()

# ✅ 설정
BASE_MODEL = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
OUTPUT_DIR = "models/first"
DATA_PATH = "/content/MI_lora_ready_800samples.json"

# ✅ QLoRA (4bit 양자화) 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ✅ 토크나이저 로딩
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# ✅ 모델 로딩 (4bit, A100 최적화)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# ✅ LoRA 설정
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base, peft_config)

# ✅ 데이터 로딩
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

dataset = Dataset.from_list(raw_data)

# ✅ 토크나이즈
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=384)

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"])

# ✅ 학습 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    save_total_limit=2,
    report_to="none",
)

# ✅ Trainer 구성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# ✅ 학습
trainer.train()

# ✅ 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ MI 파인튜닝 완료 (A100 + QLoRA 최적화)")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

<ipython-input-4-27b84600f3c6>:80: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,3.933700
20,3.823900
30,3.636700
40,3.429300
50,3.145600
60,2.892000
70,2.622700
80,2.353500
90,2.169900
100,2.058100


✅ MI 파인튜닝 완료 (A100 + QLoRA 최적화)


In [ ]:
!ls models/first


In [ ]:
# ✅ 1. 필수 패키지 설치
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -q accelerate sentencepiece

# ✅ 2. llama.cpp 최신 설치
%cd /content
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!mkdir -p build
!cmake -S . -B build -DLLAMA_BUILD_EXAMPLES=ON
!cmake --build build -j
%cd /content

# ✅ 3. LoRA 결과 디렉토리 설정 (이미 학습된 MI 모델)
lora_path = "/content/models/first"  # ✅ MI 파인튜닝된 디렉토리
merged_path = "/content/merged-first-chat"

# ✅ 4. LoRA 병합 및 FP16 저장
from peft import PeftModel, PeftConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

peft_config = PeftConfig.from_pretrained(lora_path)
base_model_path = peft_config.base_model_name_or_path

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model, lora_path)
model = model.merge_and_unload()

model.save_pretrained(merged_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
tokenizer.save_pretrained(merged_path)

# ✅ 5. GGUF로 f16 변환
!python3 /content/llama.cpp/convert_hf_to_gguf.py {merged_path}

# ✅ 6. Q4_K_M 양자화
!./llama.cpp/build/bin/quantize \
  {merged_path}/merged-mi-chat-f16.gguf \
  {merged_path}/merged-mi-chat-q4_k_m.gguf \
  q4_K_M

# ✅ 7. 결과 확인
import os
print("✅ 생성된 GGUF 목록:", os.listdir(merged_path))



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/content
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
/content/llama.cpp
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Configuring done (0.5s)
-- Generating done (0.3s)
-- Build files have been written to: /content/llama.cpp/build
[  0%] Built target build_info
[  1%] Built target llama-gemma3-cli
[  2%] Built target sha1
[  3%] Built target xxhash
[  4%] Built target llama-llava-cli
[  4%] Built target sha256
[  5%] Built target llama-minicpmv-cli
[  6%] Built target llama-qwen2

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:hf-to-gguf:Loading model: merged-first-chat
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {4096, 1024}
INFO:hf-to-gguf:blk

In [ ]:
# ✅ GGUF로 다시 변환 (정확한 경로, 이름 부여)
!python3 /content/llama.cpp/convert_hf_to_gguf.py /content/merged-first-chat \
  --outfile /content/merged-first-chat/merged-first-8.0B-chat-F16.gguf \
  --outtype f16

# ✅ Q4_K_M 양자화 (올바른 실행 파일명: llama-quantize 사용)
!./llama.cpp/build/bin/llama-quantize \
  /content/merged-first-chat/merged-first-8.0B-chat-F16.gguf \
  /content/merged-first-chat/merged-first-8.0B-chat-Q4_K_M.gguf \
  Q4_K_M

INFO:hf-to-gguf:Loading model: merged-first-chat
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {4096, 1024}
INFO:hf-to-gguf:blk

In [ ]:
from google.colab import files
files.download('/content/merged-mi-chat/merged-cbt1-chat-q4_k_m.gguf')
